# Goodreads Book Trends: SQL Analysis

## Overview

This notebook uses the completed Goodreads SQLite relational database to perform SQL-based analysis of book trends, genres, ratings, publication years, and relationships between the database entities.

The database was created in the previous notebook after the source datasets were cleaned, integrated, validated, and converted into relational tables.

## Analysis Goals

The SQL analysis focuses on several questions:

- Which genres contain the most books?
- Which genres have large enough sample sizes to support meaningful comparisons?
- How do average Goodreads ratings differ between genres?
- How has book publication volume changed over time?
- How have genre publication trends changed across different years?
- Which genres combine a substantial number of books with high average ratings?
- Can the relational database successfully connect books with authors, genres, and the integrated Sci-Fi/Fantasy dataset?

## SQL Techniques

The queries in this notebook demonstrate several SQL techniques, including:

- `SELECT`
- `JOIN`
- `GROUP BY`
- Aggregate functions such as `COUNT()` and `AVG()`
- `COUNT(DISTINCT ...)`
- `HAVING`
- `ORDER BY`
- Subqueries
- Common Table Expressions (`CTEs`)
- SQLite `PRAGMA` functionality

Pandas is used to execute the SQL queries and display the resulting datasets for analysis.

## Database Tables Used

The analysis primarily uses the following tables:

- **BOOKS** — Book-level metadata, ratings, and publication information.
- **AUTHORS** — Unique author records.
- **BOOK_AUTHORS** — Relationship between books and authors.
- **GENRES** — Unique Goodreads genres.
- **BOOK_GENRES** — Relationship between books and genres.
- **SOURCE_GENRES** — Original Goodreads source genre categories.
- **BOOK_SOURCE_GENRES** — Relationship between books and their original source datasets.
- **SCIFI_FANTASY_BOOKS** — Records from the separate Science Fiction and Fantasy dataset.
- **BOOK_DATASET_MATCHES** — Relationships between matching records across the two datasets.

## Analysis Approach

The analysis begins with basic genre and publication counts before progressing to more detailed comparisons involving ratings and genre trends.

Minimum book-count thresholds are used in several queries to reduce the influence of genres with very small sample sizes. This helps make comparisons more representative of the larger dataset.

The final queries demonstrate how the relational database can combine information from multiple tables to answer increasingly complex analytical questions.

All analysis is performed against the finalized SQLite database:

`../Data/goodreads_capstone.db`

### 1. Connect to the SQLite Database

The completed Goodreads SQLite database is opened for SQL analysis.

A database connection is established using `sqlite3`, and foreign-key enforcement is enabled to ensure that all relational queries operate against a structurally valid database.

The database path is displayed to confirm that the analysis is using the finalized capstone database.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../Data/goodreads_capstone.db")

conn = sqlite3.connect(db_path)

conn.execute("PRAGMA foreign_keys = ON;")

print("Connected to:", db_path)

Connected to: ..\Data\goodreads_capstone.db


### 2. Analyze Book Counts by Genre

This query calculates the number of unique books associated with each Goodreads genre.

The `GENRES` and `BOOK_GENRES` tables are joined through `genre_id`, and duplicate book records are prevented from affecting the results by using `COUNT(DISTINCT bg.book_id)`.

The genres are grouped and ordered from the largest number of associated books to the smallest. The first 20 results are displayed to identify the most prevalent genres in the dataset.

In [6]:
query1 = """
SELECT
    g.genre_name,
    COUNT(DISTINCT bg.book_id) AS book_count
FROM genres AS g
JOIN book_genres AS bg
    ON g.genre_id = bg.genre_id
GROUP BY g.genre_id, g.genre_name
ORDER BY book_count DESC;
"""

genre_counts = pd.read_sql_query(query1, conn)

display(genre_counts.head(20))

,genre_name,book_count
0,romance,199105
1,history,173414
2,fantasy,143890
3,childrens,116001
4,contemporary,94620
5,comics,91556
6,mystery,89657
7,audiobook,76260
8,science fiction,75022
9,young adult,71370


### 3. Identify High-Volume Genres

This query identifies genres associated with at least 100,000 unique books.

The `HAVING` clause filters the grouped results after the book counts have been calculated. This focuses the analysis on the largest genres and reduces the influence of genres with relatively few records.

The qualifying genres are ordered by book count in descending order and displayed for comparison.

In [7]:
query2 = """
SELECT
    g.genre_name,
    COUNT(DISTINCT bg.book_id) AS book_count
FROM genres AS g
JOIN book_genres AS bg
    ON g.genre_id = bg.genre_id
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT bg.book_id) >= 100000
ORDER BY book_count DESC;
"""

large_genres = pd.read_sql_query(query2, conn)

display(large_genres)

,genre_name,book_count
0,romance,199105
1,history,173414
2,fantasy,143890
3,childrens,116001


### 4. Inspect the Books Table Schema

The structure of the `BOOKS` table is inspected using SQLite's `PRAGMA table_info` command.

This confirms the available columns and their data types before queries involving book-level attributes such as publication year, ratings, and identifiers are performed.

In [8]:
books_schema = pd.read_sql_query(
    "PRAGMA table_info(books);",
    conn
)

display(books_schema)

,cid,name,type,notnull,dflt_value,pk
0,0,book_id,INTEGER,0,None,1
1,1,book_key,TEXT,1,None,0
2,2,name,TEXT,1,None,0
3,3,pub_year,INTEGER,0,None,0
4,4,star_rating,REAL,0,None,0
5,5,num_ratings,INTEGER,0,None,0
6,6,isbn_clean,TEXT,0,None,0


### 5. Compare Ratings Across High-Volume Genres

This query compares the average Goodreads rating of genres that contain at least 100,000 unique books.

A subquery first identifies the high-volume genres based on their distinct book counts. The results are then joined with the `BOOKS` table to calculate the average `star_rating` for each qualifying genre.

The query returns both the number of books and average rating for each genre, allowing genre popularity and reader ratings to be examined together.

In [9]:
query3 = """
SELECT
    g.genre_name,
    COUNT(DISTINCT bg.book_id) AS book_count,
    ROUND(AVG(b.star_rating), 2) AS average_rating
FROM genres AS g
JOIN book_genres AS bg
    ON g.genre_id = bg.genre_id
JOIN books AS b
    ON bg.book_id = b.book_id
WHERE g.genre_id IN (
    SELECT
        bg2.genre_id
    FROM book_genres AS bg2
    GROUP BY bg2.genre_id
    HAVING COUNT(DISTINCT bg2.book_id) >= 100000
)
GROUP BY g.genre_id, g.genre_name
ORDER BY average_rating DESC;
"""

genre_rating_analysis = pd.read_sql_query(query3, conn)

display(genre_rating_analysis)

,genre_name,book_count,average_rating
0,childrens,116001,3.90
1,history,173414,3.90
2,fantasy,143890,3.89
3,romance,199105,3.83


### 6. Query 4 — Average Rating by Genre

This query examines the average Goodreads rating associated with each genre.

Because books may belong to multiple genres, the query joins the `BOOKS`, `BOOK_GENRES`, and `GENRES` tables to connect book-level ratings with their associated genres.

The query also calculates the number of unique books represented in each genre. Genres with fewer than 100 books are excluded to reduce the influence of very small samples.

The results are ordered by average rating in descending order, and the top 20 genres are displayed.

This query demonstrates the use of:

- Multiple `JOIN`s
- Aggregate functions
- `GROUP BY`
- `HAVING`
- `ORDER BY`

In [5]:
query4 = """
SELECT
    g.genre_name,
    COUNT(DISTINCT b.book_id) AS book_count,
    ROUND(AVG(b.star_rating), 2) AS average_rating
FROM books AS b
JOIN book_genres AS bg
    ON b.book_id = bg.book_id
JOIN genres AS g
    ON bg.genre_id = g.genre_id
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT b.book_id) >= 100
ORDER BY average_rating DESC;
"""

genre_ratings = pd.read_sql_query(query4, conn)

display(genre_ratings.head(20))

,genre_name,book_count,average_rating
0,baha i,101,4.53
1,devotional,1527,4.33
2,lds non fiction,270,4.33
3,colouring books,188,4.31
4,prayer,1444,4.29
5,african american romance,964,4.28
6,field guides,558,4.28
7,scripture,325,4.28
8,ornithology,139,4.27
9,catholic,4079,4.24


### 7. Query 5 — Publication Trends Over Time

This query examines the number of books published in each year represented in the `BOOKS` table.

The records are grouped by `pub_year`, and the total number of books published during each year is calculated. The results are ordered chronologically to make changes in publication volume over time easier to examine.

This query provides the data needed to identify long-term publication trends within the Goodreads dataset.

In [6]:
query5 = """
SELECT
    pub_year,
    COUNT(*) AS book_count
FROM books
GROUP BY pub_year
ORDER BY pub_year;
"""

publication_trends = pd.read_sql_query(query5, conn)

display(publication_trends.head(20))

,pub_year,book_count
0,1900,711
1,1901,383
2,1902,385
3,1903,370
4,1904,410
5,1905,444
6,1906,403
7,1907,431
8,1908,439
9,1909,457


### 8. Query 6 — Genre Publication Trends Over Time

This query examines how the number of books associated with each genre changes across publication years.

The `BOOKS`, `BOOK_GENRES`, and `GENRES` tables are joined to connect publication years with their associated genres. Books are grouped by both publication year and genre, and distinct book counts are calculated for each combination.

The results are ordered chronologically by publication year, with the genres containing the most books appearing first within each year.

This query can be used to identify changes in genre prevalence over time and provides a foundation for analyzing the growth or decline of specific genres.

In [7]:
query6 = """
SELECT
    b.pub_year,
    g.genre_name,
    COUNT(DISTINCT b.book_id) AS book_count
FROM books AS b
JOIN book_genres AS bg
    ON b.book_id = bg.book_id
JOIN genres AS g
    ON bg.genre_id = g.genre_id
GROUP BY
    b.pub_year,
    g.genre_id,
    g.genre_name
ORDER BY
    b.pub_year,
    book_count DESC;
"""

genre_trends = pd.read_sql_query(query6, conn)

display(genre_trends.head(20))

,pub_year,genre_name,book_count
0,1900,classics,142
1,1900,history,109
2,1900,childrens,82
3,1900,picture books,56
4,1900,short stories,55
5,1900,reference,53
6,1900,poetry,50
7,1900,literature,46
8,1900,historical fiction,42
9,1900,philosophy,40


### 9. Query 7 — Identify Highly Rated Genres

This query identifies genres that have both a substantial number of books and a high average Goodreads rating.

A Common Table Expression (`WITH genre_statistics AS`) first calculates the number of unique books and average rating for every genre. The outer query then filters the results to include only genres with at least 1,000 books and an average rating of 4.0 or higher.

The qualifying genres are ordered by average rating in descending order.

This query demonstrates the use of:

- Common Table Expressions (`WITH`)
- Multiple `JOIN`s
- Aggregate functions
- `GROUP BY`
- Filtering calculated results
- `ORDER BY`

The minimum book-count requirement helps prevent genres with very few books from appearing highly rated simply because of a small sample size.

In [8]:
query7 = """
WITH genre_statistics AS (
    SELECT
        g.genre_id,
        g.genre_name,
        COUNT(DISTINCT b.book_id) AS book_count,
        AVG(b.star_rating) AS average_rating
    FROM genres AS g
    JOIN book_genres AS bg
        ON g.genre_id = bg.genre_id
    JOIN books AS b
        ON bg.book_id = b.book_id
    GROUP BY
        g.genre_id,
        g.genre_name
)

SELECT
    genre_name,
    book_count,
    ROUND(average_rating, 2) AS average_rating
FROM genre_statistics
WHERE book_count >= 1000
  AND average_rating >= 4.0
ORDER BY average_rating DESC;
"""

high_rated_genres = pd.read_sql_query(query7, conn)

display(high_rated_genres)

,genre_name,book_count,average_rating
0,devotional,1527,4.33
1,prayer,1444,4.29
2,catholic,4079,4.24
3,comic strips,1163,4.23
4,christian non fiction,3161,4.21
...,...,...,...
62,psychoanalysis,1199,4.01
63,western romance,2021,4.01
64,birds,2253,4.00
65,military science fiction,1222,4.00


## 10. SQL Analysis Summary

The SQL analysis phase uses the completed relational database to explore patterns within the Goodreads book dataset.

The queries examine:

- Book distribution across genres
- High-volume genres
- Average ratings by genre
- Publication trends over time
- Genre trends across publication years
- Genres with both large sample sizes and high average ratings
- Relationships between books, authors, genres, and integrated datasets

These queries demonstrate the use of relational database concepts and intermediate-to-advanced SQL techniques, including `JOIN`, `GROUP BY`, aggregate functions, `HAVING`, subqueries, Common Table Expressions (`CTEs`), and `ORDER BY`.

Together, the results provide a foundation for the project's exploratory analysis and visualizations while demonstrating how the SQLite database can be used to answer questions about Goodreads book trends.

### 11. Close Database Connection

The SQLite database connection is closed after completing the SQL analysis.

This releases the database resources and ensures the analysis notebook ends with the database connection properly terminated.

In [9]:
conn.close()

print("Database connection closed.")

Database connection closed.
